# Study 2

- Framing (Fulfilment vs. Justice)
- 3 Statements (0: A = B, 1: A > B, 2: A < B)
- 15 Endowment Cases

**Group 5 (Fulfilment) and 6 (Justice)**

| Case       | Family A   | Family B   | Deviation (A, B)   | Ratio (A, B)   |
|:----------:|:----------:|:----------:|:------------------:|:--------------:|
| **Need**   | **80**     | **120**    | -                  | -              |
|  1         |   20       |    30      | 60,  90            | 0.25, 0.25     |
|  2         |   20       |    60      | 60,  60            | 0.25, 0.50     |
|  3         |   40       |    60      | 40,  60            | 0.50, 0.50     |
|  4         |   40       |    80      | 40,  40            | 0.50, 0.67     |
|  5         |   60       |    90      | 20,  40            | 0.75, 0.75     |
|  6         |   60       |   100      | 20,  20            | 0.75, 0.83     |
|  7         |   80       |   120      |  0,   0            | 1.00, 1.00     |
|  8         |  100       |   140      | 20,  20            | 1.25, 1.17     |
|  9         |  100       |   150      | 20,  30            | 1.25, 1.25     |
| 10         |  120       |   160      | 40,  40            | 1.50, 1.33     |
| 11         |  120       |   180      | 40,  60            | 1.50, 1.50     |
| 12         |  140       |   180      | 60,  60            | 1.75, 1.50     |
| 13         |  140       |   210      | 60,  90            | 1.75, 1.75     |
| 14         |  160       |   200      | 80,  80            | 2.00, 1.67     |
| 15         |  160       |   240      | 80, 120            | 2.00, 2.00     |

## Set Up Notebook

In [ ]:
# Import packages
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import statsmodels.api as sm

## Clean Data

In [ ]:
# Load data
df = pd.read_csv('need_fulfilment_data_1_3.csv')

# Filter for groups
df = df[df['randnumber'].isin([5, 6])]

# Define case columns
case_cols = [f'case2a{i}' for i in range(1, 16)] + \
            [f'case2b{i}' for i in range(1, 16)]

# Count answers for Control Question 1
control_1_counts = df['kontrolle1'].value_counts().sort_index()

# Calculate percentages for Control Question 1
control_1_pct = df['kontrolle1'].value_counts(normalize = True).sort_index() * 100

# Display results
print("\nControl Question 1")
print("\nCounts\n", control_1_counts)
print("\nPercentages\n", control_1_pct)

# Count answers for Control Question 2
control_2_counts = df['kontrolle2b'].value_counts().sort_index()

# Calculate percentages for Control Question 2
control_2_pct = df['kontrolle2b'].value_counts(normalize = True).sort_index() * 100

# Display results
print("\nControl Question 2")
print("\nCounts\n", control_2_counts)
print("\nPercentages\n", control_2_pct)

# Create DataFrame
df_clean = df[
    (df['kontrolle1'] == 7) &
    (df['kontrolle2b'] == 1) &
    (df[case_cols].notna().any(axis = 1))
].copy()

# Count participants per group
group_counts = df_clean['randnumber'].value_counts().sort_index()

# Display results
print("\nFinal Participants per Group (Study 2)")
print(group_counts)
print(f"\nTotal clean sample size: {len(df_clean)}")

## Calculate Sociodemographics

In [ ]:
# Define labels
label_dicts = {
    'geschlecht': {
        1: 'Female',
        2: 'Non-Binary',
        3: 'Male'
    },
    'bildung': {
        1: 'Still in School',
        2: 'Lower Secondary Education',
        3: 'Intermediate Secondary Education',
        4: 'Higher Education Entrance Qualification',
        5: 'Completed Vocational Training',
        6: 'Master Craftsman',
        7: 'Bachelor',
        8: 'Master',
       11: 'Doctorate'
    },
    'bundesland': {
        1: 'Baden-Wuerttemberg',
        2: 'Bavaria',
        3: 'Berlin',
        4: 'Brandenburg',
        5: 'Bremen',
        6: 'Hamburg',
        7: 'Hesse',
        8: 'Mecklenburg-Western Pomerania',
        9: 'Lower Saxony',
        10: 'North Rhine-Westphalia',
        11: 'Rhineland-Palatinate',
        12: 'Saarland',
        13: 'Saxony',
        14: 'Saxony-Anhalt',
        15: 'Schleswig-Holstein',
        16: 'Thuringia'
    }
}

# Iterate through continuous variables
for col in ['alter', 'politik', 'wuerde', 'wohnraum', 'einkommen', 'mitbewohner']:
    
    # Ensure data is numeric
    numeric_data = pd.to_numeric(df_clean[col], errors = 'coerce')

    # Calculate metrics
    col_mean = numeric_data.mean()
    col_std  = numeric_data.std()
    col_min  = numeric_data.min()
    col_max  = numeric_data.max()

    # Display results
    print(f"\n{col}:")
    print(f"Mean: {col_mean:.2f} | Std: {col_std:.2f} | Range: {col_min} - {col_max}")

# Iterate through categorical variables
for col, mapping in label_dicts.items():
    
    # Map codes to labels
    mapped_data = df_clean[col].map(mapping)
    
    # Calculate metrics
    counts = mapped_data.value_counts()
    percentages = mapped_data.value_counts(normalize = True) * 100
    
    # Create DataFrame
    df_summary = pd.DataFrame({
        'Counts': counts,
        'Percentages (%)': percentages
    }).round(3)
    
    # Display results
    print(f"\n{col}:")
    print(df_summary)

## Calculate Descriptive Statistics

In [ ]:
# Initialize list
raw_frequency_data = []

# Iterate through cases
for i in range(1, 16):
    
    # Filter for groups
    group_5 = df_clean[df_clean['randnumber'] == 5]
    group_6 = df_clean[df_clean['randnumber'] == 6]
    
    # Extract data for cases
    data_g5 = pd.to_numeric(group_5[f'case2a{i}'], errors = 'coerce').dropna()
    data_g6 = pd.to_numeric(group_6[f'case2b{i}'], errors = 'coerce').dropna()
    
    # Get counts
    counts_g5 = data_g5.value_counts().sort_index()
    counts_g6 = data_g6.value_counts().sort_index()
    
    # Get percentages
    percentages_g5 = data_g5.value_counts(normalize = True).sort_index() * 100
    percentages_g6 = data_g6.value_counts(normalize = True).sort_index() * 100
    
    # Format strings
    def fmt(count_dict, percentages_dict, key):
        c = int(count_dict.get(key, 0))
        p = percentages_dict.get(key, 0)
        return f"{c} ({p:.1f} %)"
    
    # Store strings
    raw_frequency_data.append({
        'Case':                   i,
        'Equal (Fulfillment)' :   fmt(counts_g5, percentages_g5, 0),
        'Equal (Justice)':        fmt(counts_g6, percentages_g6, 0),
        'A Better (Fulfillment)': fmt(counts_g5, percentages_g5, 1),
        'A Better (Justice)':     fmt(counts_g6, percentages_g6, 1),
        'B Better (Fulfillment)': fmt(counts_g5, percentages_g5, 2),
        'B Better (Justice)':     fmt(counts_g6, percentages_g6, 2)
    })

# Create DataFrame
df_frequency_table = pd.DataFrame(raw_frequency_data).set_index('Case')

# Display table
df_frequency_table

## Stacked Bar Chart

In [ ]:
##################
# PREPARE CHARTS #
##################

# Define subsets
difference_cases = [2, 4, 6, 7, 8, 10, 12, 14]
ratio_cases = [1, 3, 5, 7, 9, 11, 13, 15]

# Generate column names
cols_diff_g5 = [f'case2a{i}' for i in difference_cases]
cols_diff_g6 = [f'case2b{i}' for i in difference_cases]

cols_ratio_g5 = [f'case2a{i}' for i in ratio_cases]
cols_ratio_g6 = [f'case2b{i}' for i in ratio_cases]

# Filter groups
df_g5 = df_clean[df_clean['randnumber'] == 5]
df_g6 = df_clean[df_clean['randnumber'] == 6]

# Define grayscales
bw_colors = ['#4D4D4D', '#999999', '#E6E6E6']

# Define hatches
bw_hatches = ['', '\\\\', '//']

# Define labels
legend_labels = ['Both Equal', 'A Better', 'B Better']

# Create legend that includes hatches
def create_bw_legend(ax, title):

    # Initialize list
    patches = []

    # Iterate through response categories
    for i in range(3):
        patch = mpatches.Patch(facecolor = bw_colors[i],
                               edgecolor = 'black',
                               hatch = bw_hatches[i],
                               label = legend_labels[i])
        patches.append(patch)
    
    # Render legend
    ax.legend(handles = patches, title = title, bbox_to_anchor = (1.01, 1), loc = 'upper left')

# Calculate percentages
def calculate_percentages(df_clean, columns):

    # Initialize DataFrame
    pct_df = pd.DataFrame(index = columns, columns = [0.0, 1.0, 2.0])

    # Iterate through case columns
    for col in columns:

        # Convert data to numeric and remove missing values
        vals = pd.to_numeric(df_clean[col], errors = 'coerce').dropna()
        
        # Calculate percentages
        counts = vals.value_counts(normalize = True) * 100
        
        # Ensure every answer exists
        for val in [0.0, 1.0, 2.0]:
            pct_df.loc[col, val] = counts.get(val, 0)
    
    pct_df.columns = legend_labels
    return pct_df

diff_g5_pct  = calculate_percentages(df_g5, cols_diff_g5)
diff_g6_pct  = calculate_percentages(df_g6, cols_diff_g6)
ratio_g5_pct = calculate_percentages(df_g5, cols_ratio_g5)
ratio_g6_pct = calculate_percentages(df_g6, cols_ratio_g6)


###########
# CHART 1 #
###########

# Initialize lists
plot_idx_diff, plot_data_diff = [], []

# Iterate through subset
for i, case in enumerate(difference_cases):

    # Set labels for x-axis
    plot_idx_diff.append(f'Case {case}\n(Fulfillment)')
    plot_idx_diff.append(f'Case {case}\n(Justice)')
    plot_data_diff.append(diff_g5_pct.iloc[i].values)
    plot_data_diff.append(diff_g6_pct.iloc[i].values)

# Construct DataFrame
plot_df_diff = pd.DataFrame(plot_data_diff, index = plot_idx_diff, columns = legend_labels)

# Initialize figure
fig1, ax1 = plt.subplots(figsize = (14, 7))

# Plot chart
plot_df_diff.plot(kind = 'bar', stacked = True, ax = ax1, color = bw_colors, edgecolor = 'black', alpha = 1.0)

# Get bar segments
bars = ax1.containers

# Iterate through bars
for i, container in enumerate(bars):
    
    # Loop through bar segments
    for patch in container.patches:

        # Apply hatches
        patch.set_hatch(bw_hatches[i])

# Iterate through subset
for i in range(1, len(difference_cases)):

    # Draw vertical line
    ax1.axvline(x = i * 2 - 0.5, color = 'black', linestyle = ':', linewidth = 1, alpha = 0.3)

# Set title
ax1.set_title('Cases With Equal Difference', fontsize = 14, fontweight = 'bold')

# Set label for y-axis
ax1.set_ylabel('Percentage', fontsize = 12)

# Create legend
create_bw_legend(ax1, 'Selected Answer')

# Rotate labels on x-axis
plt.xticks(rotation = 45, ha = 'right')
fig1.tight_layout()

# Export graph
plt.savefig('need_fulfilment_study_3_bar_chart_1.pdf', format = 'pdf', bbox_inches = 'tight')

# Display graph
plt.show()


###########
# CHART 2 #
###########

# Initialize lists
plot_idx_ratio, plot_data_ratio = [], []

# Iterate through subset
for i, case in enumerate(ratio_cases):
    
    # Set labels for x-axis
    plot_idx_ratio.append(f'Case {case}\n(Fulfillment)')
    plot_idx_ratio.append(f'Case {case}\n(Justice)')
    plot_data_ratio.append(ratio_g5_pct.iloc[i].values)
    plot_data_ratio.append(ratio_g6_pct.iloc[i].values)

# Construct DataFrame
plot_df_ratio = pd.DataFrame(plot_data_ratio, index = plot_idx_ratio, columns = legend_labels)

# Initialize figure
fig2, ax2 = plt.subplots(figsize = (14, 7))

# Plot chart
plot_df_ratio.plot(kind = 'bar', stacked = True, ax = ax2, color = bw_colors, edgecolor = 'black', alpha = 1.0)

# Get bar segments
bars = ax2.containers

# Iterate through bars
for i, container in enumerate(bars):
    
    # Loop through bar segments
    for patch in container.patches:
        
        # Apply hatches
        patch.set_hatch(bw_hatches[i])

# Iterate through subset
for i in range(1, len(ratio_cases)):
    
    # Draw vertical line
    ax2.axvline(x = i * 2 - 0.5, color = 'black', linestyle = ':', linewidth = 1, alpha = 0.3)

# Set title
ax2.set_title('Cases With Equal Ratio', fontsize = 14, fontweight = 'bold')

# Set label for y-axis
ax2.set_ylabel('Percentage', fontsize = 12)

# Create legend
create_bw_legend(ax2, 'Selected Answer')

# Rotate labels on x-axis
plt.xticks(rotation = 45, ha = 'right')
fig2.tight_layout()

# Export graph
plt.savefig('need_fulfilment_study_3_bar_chart_2.pdf', format = 'pdf', bbox_inches = 'tight')

# Display graph
plt.show()

## Calculate Correlations

In [ ]:
################
# CASE METRICS #
################

# Define cases (Endowment A, Need A, Endowment B, Need B)
case_definitions = {
     1: [ 20, 80,  30, 120],
     2: [ 20, 80,  60, 120],
     3: [ 40, 80,  60, 120],
     4: [ 40, 80,  80, 120],
     5: [ 60, 80,  90, 120],
     6: [ 60, 80, 100, 120],
     7: [ 80, 80, 120, 120],
     8: [100, 80, 140, 120],
     9: [100, 80, 150, 120],
    10: [120, 80, 160, 120],
    11: [120, 80, 180, 120],
    12: [140, 80, 180, 120],
    13: [140, 80, 210, 120],
    14: [160, 80, 200, 120],
    15: [160, 80, 240, 120]
}

# Initialize list
case_metrics = []

# Iterate through cases
for case_id, data in case_definitions.items():
    
    # Calculate difference of ratios
    ratio_error = abs((data[0] / data[1]) - (data[2] / data[3]))
    
    # Calculate difference of differences
    diff_error = abs((data[0] - data[1]) - (data[2] - data[3]))
    
    case_metrics.append({
        'Case': case_id,
        'Ratio_Error': ratio_error,
        'Diff_Error': diff_error
    })

# Create DataFrame
df_analysis = pd.DataFrame(case_metrics)


##########################
# RESPONSE PROBABILITIES #
##########################

# Initialize list
probabilities_g5, probabilities_g6 = [], []

# Iterate through cases
for i in range(1, 16):
    
    # Calculate probability of selecting 0 for Fulfilment
    p5 = (pd.to_numeric(df_clean[df_clean['randnumber'] == 5][f'case2a{i}'], errors = 'coerce') == 0).mean()
    probabilities_g5.append(p5)
    
    # Calculate probability of selecting 0 for Justice
    p6 = (pd.to_numeric(df_clean[df_clean['randnumber'] == 6][f'case2b{i}'], errors = 'coerce') == 0).mean()
    probabilities_g6.append(p6)

# Store in DataFrame
df_analysis['Prob_G5'] = probabilities_g5
df_analysis['Prob_G6'] = probabilities_g6


###################
# GLOBAL ANALYSIS #
###################
print("\nGlobal Analysis")

# Calculate correlation between (theoretical) error metrics and (empirical) response rates for Fulfilment
print("\nCorrelations (Fulfilment)")
print(df_analysis[['Ratio_Error', 'Diff_Error', 'Prob_G5']].corr()['Prob_G5'])

# Calculate correlation between (theoretical) error metrics and (empirical) response rates for Justice
print("\nCorrelations (Justice)")
print(df_analysis[['Ratio_Error', 'Diff_Error', 'Prob_G6']].corr()['Prob_G6'])


#####################
# SUBGROUP ANALYSIS #
#####################
print("\nSubgroup Analysis")

# Define subgroups
df_scarcity  = df_analysis[df_analysis['Case'] <= 6]
df_abundance = df_analysis[df_analysis['Case'] >= 8]

print("\nFulfillment")

# Calculate correlations for scarcity
c5_r_sc = df_scarcity[['Ratio_Error', 'Prob_G5']].corr().iloc[0, 1]
c5_d_sc = df_scarcity[['Diff_Error', 'Prob_G5']].corr().iloc[0, 1]

# Calculate correlations for abundance
c5_r_ab = df_abundance[['Ratio_Error', 'Prob_G5']].corr().iloc[0, 1]
c5_d_ab = df_abundance[['Diff_Error', 'Prob_G5']].corr().iloc[0, 1]

# Display output
print(f"Scarcity  -> Ratio: {c5_r_sc:.4f} | Diff: {c5_d_sc:.4f}")
print(f"Abundance -> Ratio: {c5_r_ab:.4f} | Diff: {c5_d_ab:.4f}")

print("\nJustice")

# Calculate correlations for scarcity
c6_r_sc = df_scarcity[['Ratio_Error', 'Prob_G6']].corr().iloc[0, 1]
c6_d_sc = df_scarcity[['Diff_Error', 'Prob_G6']].corr().iloc[0, 1]

# Calculate correlations for abundance
c6_r_ab = df_abundance[['Ratio_Error', 'Prob_G6']].corr().iloc[0, 1]
c6_d_ab = df_abundance[['Diff_Error', 'Prob_G6']].corr().iloc[0, 1]

# Display output
print(f"Scarcity  -> Ratio: {c6_r_sc:.4f} | Diff: {c6_d_sc:.4f}")
print(f"Abundance -> Ratio: {c6_r_ab:.4f} | Diff: {c6_d_ab:.4f}")

- Here, we calculated the correlation between theoretical error metrics (how far the differences or ratios deviate from a perfectly equal state) and empirical response rates (how likely it is that subjects judge A and B to be equal).
- Globally, we find **moderate negative correlations**: The larger the `ratio_error`, the less frequently participants judge the cases to be equal.
- Globally, we also find **low positive correlations**: The larger the `diff_error`, the more frequently participants judge the cases to be equal. This appears to be an artefact of our study’s design; some cases with equal ratios show the largest absolute differences (for example, take Case 1, where A is missing 90 units and B is missing 180 units, giving `diff_error = 90` and `ratio_error = 0`).
- In almost all categories, the correlation with the `ratio_error` is significantly stronger than with the `diff_error` (except for the combination of Abundance and Justice in the subgroup analysis).
- Under **scarcity**, the correlation between `ratio_error` and equality judgments increases immensely. 
- Under **abundance**, this correlation collapses immensely, indicating that participants evaluate situations fundamentally differently when needs are met.
- While the global difference between Fulfilment and Justice appears to be only marginal, this is due to a masking effect by the abundance cases. When looking specifically at **scarcity**, the framing effect becomes visible: The negative correlation with the `ratio_error` is notably stronger in the Justice framing than in the Fulfilment framing.

## Perform OLS Regressions

In [ ]:
#######################
# PREPARE REGRESSIONS #
#######################

# Define control variables
control_vars = ['wuerde', 'mitbewohner', 'wohnraum', 'alter', 'geschlecht', 'einkommen', 'bildung', 'bundesland', 'politik']

# Create identifier for each participant
df_clean['subject_id'] = df_clean.index

# Generate list of case columns
case_cols = [f'case2a{i}' for i in range(1, 16)] + [f'case2b{i}' for i in range(1, 16)]

# Define columns to keep during reshaping
cols_to_keep = ['subject_id', 'randnumber'] + control_vars

# Reshape data from wide to long format
df_long = pd.melt(df_clean,
                  id_vars = cols_to_keep,
                  value_vars = case_cols,
                  var_name = 'Case_String',
                  value_name = 'Response'
                 )

# Ensure numeric types for the response variable
df_long['Response'] = pd.to_numeric(df_long['Response'], errors = 'coerce')

# Remove observations with missing responses
df_long = df_long.dropna(subset = ['Response'])

# Create binary dependent variable (0: A ≠ B, 1: A = B)
df_long['Is_Equal'] = (df_long['Response'] == 0).astype(int)

# Extract case number from string
df_long['Case'] = df_long['Case_String'].str.extract(r'case2[ab](\d+)').astype(int)

# Merge data with error metrics
df_regression = df_long.merge(df_analysis[['Case', 'Ratio_Error', 'Diff_Error']], on = 'Case')

# Create framing dummy (0: Fulfilment, 1: Justice)
df_regression['Justice_Framing'] = (df_regression['randnumber'] == 6).astype(int)

# Drop rows with missing values in control variables to ensure consistent sample size
df_regression_clean = df_regression.dropna(subset = control_vars).copy()

# Define continuous variables for standardization
vars_to_std = ['Ratio_Error', 'Diff_Error', 'alter', 'politik']

# Apply z-transformation
for var in vars_to_std:
    df_regression_clean[f'{var}_z'] = (df_regression_clean[var] - df_regression_clean[var].mean()) / df_regression_clean[var].std()

# Define dependent variable
y = df_regression_clean['Is_Equal']

# Identify cluster variable for robust standard errors
cluster_id = df_regression_clean['subject_id']


###########
# MODEL 1 #
###########

# Define predictors
predictors_m1 = ['Ratio_Error_z', 'Diff_Error_z', 'Justice_Framing']

# Add constant
X1 = sm.add_constant(df_regression_clean[predictors_m1])

# Fit model
model_1 = sm.OLS(y, X1).fit(cov_type = 'cluster', cov_kwds = {'groups': cluster_id})

# Display results
print("\nModel 1: Main Effects")
print(model_1.summary())


###########
# MODEL 2 #
###########

# Define control variables
control_vars_new = ['wuerde', 'mitbewohner', 'wohnraum', 'alter_z', 'geschlecht', 'einkommen', 'bildung', 'bundesland', 'politik_z']

# Define predictors
predictors_m2 = ['Ratio_Error_z', 'Diff_Error_z', 'Justice_Framing'] + control_vars_new

# Add constant
X2 = sm.add_constant(df_regression_clean[predictors_m2])

# Fit model
model_2 = sm.OLS(y, X2).fit(cov_type = 'cluster', cov_kwds = {'groups': cluster_id})

# Display results
print("\nModel 2: Main Effects and Control Variables")
print(model_2.summary())


###########
# MODEL 3 #
###########

# Define dummy (0: Exact Supply/Oversupply, 1: Undersupply)
df_regression_clean['Undersupply'] = (df_regression_clean['Case'] <= 6).astype(int)

# Calculate two-way interaction terms
df_regression_clean['Ratio_x_Framing'] = df_regression_clean['Ratio_Error_z'] * df_regression_clean['Justice_Framing']
df_regression_clean['Diff_x_Framing']  = df_regression_clean['Diff_Error_z']  * df_regression_clean['Justice_Framing']

df_regression_clean['Ratio_x_Under']   = df_regression_clean['Ratio_Error_z']   * df_regression_clean['Undersupply']
df_regression_clean['Diff_x_Under']    = df_regression_clean['Diff_Error_z']    * df_regression_clean['Undersupply']
df_regression_clean['Framing_x_Under'] = df_regression_clean['Justice_Framing'] * df_regression_clean['Undersupply']

# Calculate three-way interaction terms
df_regression_clean['Ratio_x_Framing_x_Under'] = df_regression_clean['Ratio_x_Framing'] * df_regression_clean['Undersupply']
df_regression_clean['Diff_x_Framing_x_Under']  = df_regression_clean['Diff_x_Framing']  * df_regression_clean['Undersupply']

# Define predictors
predictors_m3 = [
    'Ratio_Error_z', 'Diff_Error_z', 'Justice_Framing', 'Undersupply',
    'Ratio_x_Framing', 'Diff_x_Framing',
    'Ratio_x_Under', 'Diff_x_Under',
    'Framing_x_Under',
    'Ratio_x_Framing_x_Under', 'Diff_x_Framing_x_Under'
] + control_vars_new

# Add constant
X3 = sm.add_constant(df_regression_clean[predictors_m3])

# Fit model
model_3 = sm.OLS(y, X3).fit(cov_type = 'cluster', cov_kwds = {'groups': cluster_id})

# Display results
print("\nModel 3: Main Effects, Interaction Terms, and Control Variables")
print(model_3.summary())

- Regression analysis reveals that undersupply is a massive driver of perceived equality. Simply being in a state of undersupply (Family A receiving less than their need) decreases the likelihood of participants evaluating the distribution as equal.
- When needs are met, we find that both `Ratio_Error` and `Diff_Error` are highly significant negative drivers.
- The framing effect is conditional on scarcity. The significant three-way interaction `Ratio_x_Framing_x_Under` indicates that the Justice framing alters participants' sensitivity in cases of undersupply. When dealing with scarcity, the Justice framing makes participants significantly more strict toward ratio errors. In cases with oversupply, framing makes no significant difference.
- Demographic controls show no significant influence on perceived equality. The only exception is `alter_z`, suggesting that older participants are slightly less likely to judge the distributions as equal.